![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System


Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

### 1.1 Background

Food prices in Kenya move sharply and unevenly across markets. Staples such as maize and beans respond to harvest cycles, rainfall, and supply conditions, but the people most exposed to that movement rarely see it coming.

Farmers face a timing decision every season: sell early and risk missing a better price, or hold and risk a drop. Traders and institutions face the same problem from the other side, deciding when to buy, store, or release stock.

None of this is a data problem in the strict sense. Kenya already publishes market price data through WFP and HDX. What is missing is a way to turn that historical record, combined with weather data, into a forward-looking view: where prices are likely headed, and when a market is starting to move outside its own normal range.

### 1.2 Problem Statement

Farmers, traders, and food security institutions largely make decisions on current or historical prices, not forecasts. That gap shows up as poor sell/buy timing, avoidable inventory risk, and institutional responses that arrive only after a shortage or price spike is already visible.

The core problem is not a lack of data. It is the absence of a system that combines historical prices with external drivers like weather to produce forecasts and early warnings at the market level.

### 1.3 Business Objectives

1. **Forward-looking price visibility** — forecast commodity prices two to three months ahead, per market.
2. **Early shock detection** — flag when actual prices start diverging meaningfully from what is expected.
3. **Accessible market intelligence** — surface forecasts and trends through a dashboard usable by non-technical stakeholders.
4. **Decision support at the institutional level** — provide a quantitative signal that can inform reserves, subsidies, procurement, and humanitarian response.
5. **A reproducible pipeline** — one automated path from raw data to forecast to dashboard, not a one-off analysis.

### 1.4 Stakeholder Analysis

**Smallholder farmers and cooperatives** — decide when and how much to sell; forecasts give visibility into where prices are headed before committing.

**Traders and market intermediaries** — manage inventory timing; forecasts reduce the risk of buying near a peak or holding through a decline.

**County agricultural offices and NDMA** — need early, localized signals of price stress before they escalate into broader food security concerns.

**NGOs and humanitarian organizations** — plan procurement and cash-based interventions; earlier visibility protects purchasing power against rising prices.

**National Cereals and Produce Board** — makes reserve, procurement, and stabilization decisions where a forecast is one more input.

**Urban consumers and low-income households** — most exposed to staple price swings; benefit indirectly through institutions that plan ahead on their behalf.

**Food processors and millers** — need predictable input costs for production and pricing decisions.

Across all of these, the shared need is the same: timely, market-specific signal about where food prices are heading, not just where they have been.

### 1.5 Business Success Criteria

This is judged successful from a business standpoint if:

- Forecasts and alerts give stakeholders information they did not already have from watching current prices.
- The anomaly detector surfaces genuine shocks without burying users in false alarms.
- A non-technical user can pick a market and commodity and immediately understand the expected price and current alert status.
- The pipeline can be refreshed with new data without significant manual rework.
- The system runs on public, freely accessible data sources, so it is not tied to a paid feed that could disappear.

### 1.6 Data Mining Goals

The technical work behind those business goals breaks into five tasks:

1. Build a **naive persistence baseline**, since any model must justify itself against it.
2. Build a **Prophet model** as the first real forecasting approach.
3. Build an **LSTM model**, both per-pair and pooled across markets and commodities, for comparison.
4. Join WFP Kenya food price data with NASA POWER weather data by market location and date.
5. Build a **residual-based anomaly detector** to flag unusual price movements.

All of this is meant to run across the full shortlist of market-commodity pairs, not a single illustrative series, since the dashboard needs coverage across markets to be useful. Results are compared using MAE and MAPE, and delivered through a Streamlit dashboard.

### 1.7 Data Mining Success Criteria

This is judged successful from a technical standpoint if:

- Forecast accuracy is measured with MAE and MAPE, consistently, across models.
- A model's usefulness is judged against the **naive persistence baseline**, not an arbitrary fixed error threshold. Beating the baseline is the bar; the baseline's actual value is established once it is computed in Modelling, not assumed here.
- The price and weather datasets join with minimal data loss.
- The anomaly detector shows a meaningfully higher flag rate during documented historical shocks than its own baseline flag rate.
- The pipeline runs end-to-end with minimal manual intervention, and produces the **same result on repeated runs** — which means every stochastic step (model initialization, training) needs to be seeded, not left to chance.
- The deployed dashboard loads reliably and reflects the pipeline's latest output.

### 1.8 Hypotheses Guiding the Analysis

- **H1:** Maize prices follow a seasonal pattern tied to harvest periods.
- **H2:** Rainfall has a measurable lagged relationship with future commodity prices.
- **H3:** Price patterns differ meaningfully across markets.
- **H4:** Maize and bean prices move together, consistent with their role as substitute staples.
- **H5:** Price volatility increases during drought periods.
- **H6:** Wholesale price changes are reflected in retail prices within one to two weeks.

These are tested, not assumed, through the EDA, feature engineering, and modelling sections that follow.

## 2. Data Understanding

This project draws on two datasets: historical food prices from WFP, and daily weather observations from NASA POWER. The goal here is to establish what each dataset contains, how it is structured, and where its quality limits are, before any cleaning or joining takes place.

### 2.1 Source of Data

| Dataset | Source | Purpose |
|---|---|---|
| WFP Kenya Food Prices | HDX (Humanitarian Data Exchange) | Historical commodity prices — the target variable |
| NASA POWER | NASA POWER API | Daily weather observations — external explanatory variables |



In [ ]:
# core libraries and notebook display settings
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# record the date the data snapshot was pulled, for reproducibility
extraction_date = datetime.now().strftime("%Y-%m-%d")
print(f"Data extraction date recorded: {extraction_date}")

In [ ]:
DATA_URL = (
    "https://data.humdata.org/dataset/"
    "e0d3fba6-f9a2-45d7-b949-140c455197ff/"
    "resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/"
    "download/wfp_food_prices_ken.csv"
)
FILENAME = "wfp_food_prices_ken.csv"


def load_food_prices():
    """
    Load the Kenya food prices dataset.

    Checks, in order: a Kaggle input copy, a saved local copy,
    then falls back to the source URL.

    Returns
    -------
    pandas.DataFrame
        Raw Kenya food prices dataset.
    """
    kaggle_root = "/kaggle/input"

    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if FILENAME in files:
                kaggle_path = os.path.join(root, FILENAME)
                try:
                    prices_raw = pd.read_csv(kaggle_path)
                    print(f"Loaded dataset from Kaggle: {kaggle_path}")
                    return prices_raw
                except Exception as error:
                    print(f"Kaggle file could not be read: {error}")

    if os.path.exists(FILENAME):
        try:
            prices_raw = pd.read_csv(FILENAME)
            print(f"Loaded local dataset: {FILENAME}")
            return prices_raw
        except Exception as error:
            print(f"Local file could not be read: {error}")

    print("Dataset not found locally. Trying the source URL...")
    try:
        response = requests.get(DATA_URL, timeout=30)
        response.raise_for_status()
        with open(FILENAME, "wb") as file:
            file.write(response.content)
        prices_raw = pd.read_csv(FILENAME)
        print(f"Dataset downloaded and saved as '{FILENAME}'")
        return prices_raw
    except Exception as error:
        print(f"Download failed: {error}")

    if os.path.exists(FILENAME):
        print("Using the previously saved local dataset.")
        return pd.read_csv(FILENAME)

    raise FileNotFoundError(
        "Could not load the Kenya food prices dataset. "
        "Check your internet connection or provide a local copy."
    )

In [ ]:
prices_raw = load_food_prices()
prices_raw.head()

In [ ]:
# drop the units/description row if present, then fix dtypes
def clean_price_data(prices_raw):
    """Clean and convert data types in the raw price dataset."""
    if prices_raw.iloc[0].astype(str).str.startswith("#").any():
        prices = prices_raw.iloc[1:].reset_index(drop=True)
    else:
        prices = prices_raw.copy()

    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    prices["price"] = pd.to_numeric(prices["price"], errors="coerce")

    if "usdprice" in prices.columns:
        prices["usdprice"] = pd.to_numeric(prices["usdprice"], errors="coerce")

    return prices


prices = clean_price_data(prices_raw)
print(f"Rows: {prices.shape[0]}, Columns: {prices.shape[1]}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")

In [ ]:
# confirm the NASA POWER API is reachable and returns the expected structure
def get_weather_sample(latitude, longitude, start="20240101", end="20240131"):
    """Fetch a short sample of daily weather data for one point."""
    power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M",
        "community": "ag",
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    response = requests.get(power_url, params=params, timeout=30)
    response.raise_for_status()
    print(f"status code {response.status_code}")
    return response.json()["properties"]["parameter"]


nairobi_lat, nairobi_lon = -1.2864, 36.8172
sample_params = get_weather_sample(latitude=nairobi_lat, longitude=nairobi_lon)
print("Parameters returned:", list(sample_params.keys()))

### 2.2 Dataset Description

The two datasets sit at different grains and are joined on market location and date.

**WFP food prices** — one row per commodity, market, and date. Each row is a single reported price for a specific pairing of item, place, and time, at either retail or wholesale level.

**NASA POWER weather** — one row per day for a given latitude/longitude point. There is no market identifier in the weather data itself; the link back to a market happens through the market's coordinates, which are already present in the price dataset.

This means the join key is not a shared ID column but a derived one: market coordinates plus date, matched to the nearest weather point.

### 2.3 Feature/Data Dictionary

**WFP Kenya Food Prices**

| Column | Type | Description | Example | Role |
|---|---|---|---|---|
| `date` | datetime64 | Date of the price observation | 2006-01-15 | Temporal |
| `admin1` | string | Top-level administrative region | Coast, Eastern | Metadata |
| `admin2` | string | Second-level administrative region | Mombasa, Kitui | Metadata |
| `market` | string | Name of the local food market | Mombasa, Kitui | Identifier / grouping |
| `market_id` | integer | Unique market identifier | 191 | Identifier |
| `latitude` | float | Market latitude | -4.05 | Geographic (join key) |
| `longitude` | float | Market longitude | 39.67 | Geographic (join key) |
| `category` | string | Broad commodity group | cereals and tubers | Metadata |
| `commodity` | string | Specific food item | Maize (white), Beans | Identifier / grouping |
| `commodity_id` | integer | Unique commodity identifier | 67 | Identifier |
| `unit` | string | Unit the price is quoted in | KG, 90 KG | Scoping |
| `priceflag` | string | Source data state | actual | Metadata |
| `pricetype` | string | Trade level | Wholesale, Retail | Scoping |
| `currency` | string | Currency | KES | Metadata |
| **`price`** | float | Price in Kenyan Shillings | 1480.00 | **Target variable (raw)** |
| `usdprice` | float | Price in USD equivalent | 20.58 | Not used |

`price` is the raw form of the target. It is standardized into `price_per_kg` during Data Preparation, once unit and trade-level scoping are resolved, and that standardized value is what the models are actually trained to predict.

**NASA POWER Weather**

| Parameter | Type | Description | Unit | Role |
|---|---|---|---|---|
| `T2M` | float | Average air temperature at 2m | °C | Predictor (used) |
| `T2M_MAX` | float | Maximum air temperature at 2m | °C | Not used |
| `T2M_MIN` | float | Minimum air temperature at 2m | °C | Not used |
| `PRECTOTCORR` | float | Bias-corrected total precipitation | mm/day | Predictor (used) |
| `RH2M` | float | Relative humidity at 2m | % | Not used |

Rainfall (`PRECTOTCORR`) and temperature (`T2M`) are the two variables this project carries forward as predictors, since they have the clearest plausible link to agricultural output and price movement.

### 2.4 Initial Data Quality Assessment

In [ ]:
# column-level summary: dtype, uniqueness, missingness
data_dictionary = pd.DataFrame({
    "column": prices.columns,
    "dtype": [str(prices[col].dtype) for col in prices.columns],
    "n_unique": [prices[col].nunique() for col in prices.columns],
    "n_missing": [prices[col].isna().sum() for col in prices.columns],
    "pct_missing": [(prices[col].isna().mean() * 100).round(2) for col in prices.columns],
})
data_dictionary

In [ ]:
prices[["price", "usdprice"]].describe()

In [ ]:
grain_columns = ["date", "market", "commodity", "pricetype"]
duplicate_count = prices.duplicated(subset=grain_columns).sum()

print(f"Duplicate rows at (date, market, commodity, pricetype) grain: {duplicate_count}")
print(f"Unique markets: {prices['market'].nunique()}")
print(f"Unique commodities: {prices['commodity'].nunique()}")
print(f"Unique admin1 regions: {prices['admin1'].nunique()}")

In [ ]:
# confirm lat/lon exist directly in the price data, and check coverage
coordinate_columns = [col for col in prices.columns if "lat" in col.lower() or "lon" in col.lower()]
coord_check = prices.groupby("market")[coordinate_columns].nunique()

missing_coords = prices[prices["latitude"].isna()]["market"].unique()

print("Coordinate columns found:", coordinate_columns)
print(f"Markets with missing coordinates: {len(missing_coords)}")
print(missing_coords)

Latitude and longitude are already present in the price dataset itself, one fixed pair per market, so no separate market-reference file is needed for the weather join.

**Summary of data quality findings:**

- **Commodity labelling:** Maize appears under five separate labels (`Maize (white)`, `Maize`, `Maize flour`, `Maize (white, dry)`, `Maize flour (white)`). These need consolidation before analysis, since treating them as unrelated commodities would understate maize's true market coverage.
- **Duplicates:** none at the (date, market, commodity, pricetype) grain.
- **Coordinates:** complete for 225 of 226 markets. Hola (Tana River) is the sole exception and will be excluded from the weather join rather than imputed.
- **Missingness:** confined to `admin1`, `admin2`, `latitude`, `longitude` — all 62 rows tied to the same single market.
- **Coverage:** the price series spans January 2006 to August 2026, a long enough window to support both long-history and short-history modelling tracks.
- **Units and trade level:** `unit` and `pricetype` are not yet standardized (13 distinct units, retail and wholesale mixed) — this is scoped and resolved explicitly in Data Preparation, not here.

The dataset is fit for the next stage. The open items above are cleanup work, not blockers.

## 3. Data Cleaning and Data Preparation

This phase turns the raw price table into a modelling-ready dataset. Commodity labels are kept as reported rather than merged, since the source data already distinguishes products, such as raw grain from flour, that behave differently in price.

Scope, coverage, and quality decisions are made explicit here so that what enters modelling is a deliberate shortlist, not whatever happened to survive incidental filtering.

### 3.1 Initial Cleaning

Row-level type coercion (parsing `date`, `price`, and `usdprice`) already happened in Data Understanding, since it was needed to profile the data honestly. The first substantive preparation step is reviewing what the `commodity` field actually contains: 51 distinct labels, some of them raw or staple products, others processed derivatives such as flour or meal. A visibility flag distinguishes the two groups. It is not used to merge products together — commodity identity is preserved throughout.

In [ ]:
commodity_overview = prices["commodity"].value_counts()
is_processed = prices["commodity"].str.contains("flour|meal|powder", case=False, na=False)
prices["is_processed"] = is_processed

print(f"Total distinct commodity labels: {prices['commodity'].nunique()}")
print(f"Processed or derivative product rows: {is_processed.sum()}")
commodity_overview.head(10)

### 3.2 Data Type Handling

`price` is only meaningful once its unit is known — 1,480 KES means something different for a kilogram of maize than for a 90 kilogram sack. Every distinct unit in the dataset is mapped to a kilogram-equivalent multiplier, covering all weight-based units observed, not just the ones maize happens to use.

In [ ]:
unit_counts = prices["unit"].value_counts()
print(unit_counts)

In [ ]:
unit_to_kg = {
    "KG": 1,
    "90 KG": 90,
    "64 KG": 64,
    "50 KG": 50,
    "26 KG": 26,
    "126 KG": 126,
    "13 KG": 13,
    "200 G": 0.2,
    "400 G": 0.4,
}

prices["kg_equivalent"] = prices["unit"].map(unit_to_kg)

### 3.3 Unit Standardization

With a kilogram-equivalent defined for every weight-based unit, price becomes directly comparable across records as `price_per_kg`. Units with no weight equivalent — sold by volume or by count — are left unconverted rather than force-fit, and identified explicitly here so the next step can make a clean scope decision about them.

In [ ]:
prices["price_per_kg"] = prices["price"] / prices["kg_equivalent"]

unmapped_units = prices[prices["kg_equivalent"].isna()]["unit"].unique()
print(f"Unmapped units, genuinely non-weight based: {list(unmapped_units)}")
print(f"Rows converted to price per kg: {prices['price_per_kg'].notna().sum()} out of {len(prices)}")

### 3.4 Commodity and Scope Selection

Two categories of commodity are removed before completeness or coverage is ever computed, rather than left to fail silently at the price-per-kg step several sections later.

Fuel (diesel, kerosene, petrol-gasoline) is out of scope — this is a food price project, and these three are present only because the WFP dataset tracks a broader commodity basket than food. Commodities priced in a unit with no kilogram equivalent — litres, millilitres, or a bare count — are excluded on measurement grounds, not as a data quality failure. Milk in all four varieties, vegetable oil, bananas, kale, and cabbage fall into this group: legitimate food commodities, but priced in a way a per-kilogram index was never built to handle. Filtering by unit compatibility rather than hardcoding these names also protects against any other commodity sharing the same problem.

In [ ]:
FUEL_COMMODITIES = ["Fuel (diesel)", "Fuel (kerosene)", "Fuel (petrol-gasoline)"]

excluded_units = sorted(set(prices["unit"]) - set(unit_to_kg))
unit_excluded_commodities = prices[prices["unit"].isin(excluded_units)]["commodity"].unique()

print(f"Excluding {len(FUEL_COMMODITIES)} fuel commodities on scope grounds")
print(f"Excluding {len(unit_excluded_commodities)} commodities priced in a non-weight unit: {list(unit_excluded_commodities)}")

prices_scoped = prices[
    ~prices["commodity"].isin(FUEL_COMMODITIES) &
    prices["unit"].isin(unit_to_kg)
].copy()

print(f"Rows after scope filtering: {len(prices_scoped)} out of {len(prices)}")

### 3.5 Retail Price Selection

Retail is selected as the primary modelling series — a deliberate scoping decision, not an oversight. Retail is the price point that smallholder farmers, cooperatives, and urban consumers, the majority of the stakeholder groups from Section 1, actually transact on. Wholesale forecasting, more relevant to traders, NCPB, and millers, is a natural extension of the same pipeline: wholesale rows remain intact in the source data and need no new collection, only rerunning completeness and modelling against `pricetype == "Wholesale"`. That is documented here as a defined next phase, not an unaddressed gap.

In [ ]:
retail = prices_scoped[prices_scoped["pricetype"] == "Retail"].copy()
wholesale_rows = prices_scoped[prices_scoped["pricetype"] == "Wholesale"]

print(f"Retail rows: {len(retail)}")
print(f"Wholesale rows: {len(wholesale_rows)}")

### 3.6 Missing Data / Completeness

Completeness is computed for every market-commodity pair at once, against the full months available in the retail dataset, before any threshold is chosen. This gives an honest picture of coverage before the shortlist is narrowed.

In [ ]:
coverage = (
    retail
    .groupby(["market", "commodity"])["date"]
    .nunique()
    .reset_index(name="months_reported")
)

total_months_available = retail["date"].nunique()
coverage["completeness_pct"] = (coverage["months_reported"] / total_months_available * 100).round(1)

print(f"Total market-commodity pairs: {len(coverage)}")
for threshold in [30, 40, 50, 60, 70]:
    qualifying = coverage[coverage["completeness_pct"] >= threshold]
    print(f"At {threshold}% threshold: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

Measured against the full dataset span, almost nothing qualifies. That is because most series only began consistent reporting in late 2023 — a fixed-span measurement unfairly penalizes pairs that started later but have reported reliably since. Coverage needs to be measured against each pair's *own* active reporting window instead.

In [ ]:
reporting_span = (
    retail
    .groupby(["market", "commodity"])["date"]
    .agg(first_reported="min", last_reported="max", months_reported="nunique")
    .reset_index()
)

reporting_span["active_months"] = (
    (reporting_span["last_reported"].dt.year - reporting_span["first_reported"].dt.year) * 12
    + (reporting_span["last_reported"].dt.month - reporting_span["first_reported"].dt.month)
    + 1
)

reporting_span["completeness_pct_fair"] = (
    reporting_span["months_reported"] / reporting_span["active_months"] * 100
).round(1)

for threshold in [50, 60, 70, 80, 90]:
    qualifying = reporting_span[reporting_span["completeness_pct_fair"] >= threshold]
    print(f"At {threshold}% fair completeness: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

### 3.7 Historical Coverage

Sufficient observations are not the same as sufficient history. Detecting a genuine seasonal pattern requires seeing it repeat across multiple years, so pairs are classified by both fair completeness and years of active history. Deeper-history pairs are routed to Prophet, which needs multiple seasonal cycles to be reliable; shorter but still-consistent pairs are routed to LSTM instead of being discarded.

In [ ]:
reporting_span["years_active"] = (
    (reporting_span["last_reported"] - reporting_span["first_reported"]).dt.days / 365.25
).round(1)

MIN_YEARS_ACTIVE = 1.0

long_history = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= 3)
].copy()

recent_only = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= MIN_YEARS_ACTIVE) &
    (reporting_span["years_active"] < 3)
].copy()

insufficient_data = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] < MIN_YEARS_ACTIVE)
]

print(f"Long history pairs (3+ years): {len(long_history)}")
print(f"Recent only pairs (1 to 3 years): {len(recent_only)}")
print(f"Insufficient data pairs (under 1 year, excluded): {len(insufficient_data)}")

Together, the long-history and recent-only groups define the full modelling scope. Each pair keeps its original commodity label, market, and price type — nothing is merged.

In [ ]:
long_history["model_track"] = "prophet"
recent_only["model_track"] = "lstm"

shortlist = pd.concat([long_history, recent_only], ignore_index=True)[["market", "commodity", "model_track"]]

print(f"Total shortlisted market-commodity pairs: {len(shortlist)}")
print(f"Unique markets: {shortlist['market'].nunique()}")
print(f"Unique commodities: {shortlist['commodity'].nunique()}")

In [ ]:
retail["kg_equivalent"] = retail["unit"].map(unit_to_kg)
retail["price_per_kg"] = retail["price"] / retail["kg_equivalent"]

modeling_data = retail.merge(shortlist, on=["market", "commodity"], how="inner")
modeling_data = modeling_data[modeling_data["price_per_kg"].notna()].copy()

print(f"Modelling-ready rows: {len(modeling_data)}")

### 3.8 Outlier Handling

Outlier bounds are computed on the standardized `price_per_kg` field, separately per commodity — pooling different commodities together would produce meaningless bounds, the same mistake an unstandardized check would make.

In [ ]:
def flag_outliers(group):
    q1, q3 = group["price_per_kg"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return group[(group["price_per_kg"] < lower) | (group["price_per_kg"] > upper)]

outliers_by_commodity = (
    modeling_data
    .groupby("commodity", group_keys=True)
    .apply(flag_outliers, include_groups=False)
    .reset_index(level=0)
)

print(f"Total outliers flagged: {len(outliers_by_commodity)} out of {len(modeling_data)}")
print(outliers_by_commodity["commodity"].value_counts())

Manual inspection of the salt outliers shows a tight, plausible cluster around 90–115 KES/kg, with a tail toward 275 KES/kg concentrated in refugee-camp markets such as Kakuma, Dadaab, and Kalobeyei, where transport and supply chain costs run higher. These are genuine prices, not conversion errors — and every flagged outlier, across every commodity, is retained rather than removed, since the anomaly detection layer built later is designed specifically to act on this kind of divergence.

In [ ]:
salt_outliers = outliers_by_commodity[outliers_by_commodity["commodity"] == "Salt"]
salt_outliers[["market", "date", "price", "unit", "price_per_kg"]]

An IQR flag alone can't tell a genuine market shift, where prices settle at a new level and stay there, from a transient spike that reverts, or a one-off entry error. Each flagged outlier is classified by comparing the price level immediately after the flagged date against the baseline immediately before it. This same before/after comparison logic is the direct precursor to the residual-based anomaly detector built in Section 7.

In [ ]:
def classify_outlier(row, data, window=2):
    series = data[(data["market"] == row["market"]) & (data["commodity"] == row["commodity"])].sort_values("date")
    match = series[series["date"] == row["date"]]
    if match.empty:
        return "unknown"
    pos = series.index.get_loc(match.index[0])
    before = series.iloc[max(0, pos - window):pos]["price_per_kg"]
    after = series.iloc[pos + 1: pos + 1 + window]["price_per_kg"]
    if before.empty or after.empty:
        return "insufficient surrounding data"
    baseline = before.mean()
    reverted = abs(after.mean() - baseline) < abs(row["price_per_kg"] - baseline) * 0.5
    return "transient spike" if reverted else "persistent shift"

outliers_by_commodity["classification"] = outliers_by_commodity.apply(
    lambda row: classify_outlier(row, modeling_data), axis=1
)

print(outliers_by_commodity["classification"].value_counts())

No flagged outlier is removed from the modelling dataset. Transient spikes are retained as genuine historical events, reserved as informal validation cases for the anomaly detector built later. Persistent shifts are retained too, though a sample was manually reviewed to rule out unit or entry errors masquerading as real shifts, since the two produce an identical pattern on this test. Rows with insufficient surrounding data are kept but not treated as evidence either way.

### 3.9 Dataset Integration

With the shortlist finalized, daily rainfall and temperature are retrieved from the NASA POWER API for every shortlisted market's coordinates, across the full date range needed for modelling. Weather is recorded daily but price is recorded monthly, so weather is aggregated to a monthly grain per market before joining — rainfall summed, temperature averaged, matching how each variable naturally accumulates over a month.

In [ ]:
market_coords = modeling_data[["market", "latitude", "longitude"]].drop_duplicates()
weather_records = []
power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"




for _, row in market_coords.iterrows():
    params_loop = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "ag",
        "longitude": row["longitude"],
        "latitude": row["latitude"],
        "start": "20060101",
        "end": "20260815",
        "format": "JSON",
    }
    resp = requests.get(power_url, params=params_loop, timeout=60)
    if resp.status_code == 200:
        param_data = resp.json()["properties"]["parameter"]
        df_market = pd.DataFrame({
            "date": pd.to_datetime(list(param_data["T2M"].keys()), format="%Y%m%d"),
            "temperature": list(param_data["T2M"].values()),
            "rainfall": list(param_data["PRECTOTCORR"].values()),
        })
        df_market["market"] = row["market"]
        weather_records.append(df_market)
    time.sleep(1)

weather_all = pd.concat(weather_records, ignore_index=True)

print(f"Markets retrieved: {weather_all['market'].nunique()} out of {len(market_coords)}")
print(f"Total daily weather rows: {len(weather_all)}")

In [ ]:
weather_monthly = (
    weather_all
    .set_index("date")
    .groupby("market")
    .resample("ME")
    .agg({"rainfall": "sum", "temperature": "mean"})
    .reset_index()
)

print(f"Monthly weather rows: {len(weather_monthly)}")
print(f"Markets represented: {weather_monthly['market'].nunique()}")

Cleaned monthly prices are joined to the lagged monthly weather on market and date. The weather table's own `date` column is dropped before merging — both tables otherwise carry a column named `date`, which would silently become `date_x`/`date_y` and break every downstream step that references `date` directly.

In [ ]:
modeling_data["date_month"] = modeling_data["date"].values.astype("datetime64[M]")
weather_monthly["date_month"] = weather_monthly["date"].values.astype("datetime64[M]")
weather_features = weather_monthly.drop(columns=["date"])

master = modeling_data.merge(
    weather_features,
    on=["market", "date_month"],
    how="left"
)

print(f"Master table rows: {len(master)}")
print(f"Rows with matched weather data: {master['rainfall'].notna().sum()}")

### 3.10 Data Preparation Summary

Three structural decisions shaped this phase. Commodity labels were kept distinct rather than merged, since WFP records raw grain and flour, for instance, as genuinely different products with different price behavior. Completeness was measured against each pair's own active reporting window rather than the full dataset span, since most series only began consistent reporting in late 2023, and a fixed-span measure would have unfairly penalized everything that started later. Units were standardized to a common price-per-kilogram basis, with weight-based units converted and genuinely non-weight units left out of weight-based analysis entirely.

The result is a shortlist of 133 market-commodity pairs across 26 markets and 14 commodities: 100 pairs with three or more years of history routed to Prophet, and 33 shorter-history pairs routed to LSTM. Outliers were identified per commodity and retained rather than removed, since the anomaly detection layer is designed to act on genuine divergence, not treat it as noise. Weather was retrieved, aggregated to monthly, lagged by three and four months, and joined to price, producing a master table of 6,077 rows with a 98.98% weather match rate — the one shortlisted market without a match, Hola (Tana River), lacks coordinates in the source data.

Whether that three- and four-month rainfall lag actually holds up is tested properly in Exploratory Data Analysis, once weather and price share a genuinely matched market and date for every row.

## 4. Exploratory Data Analysis

With scope, units, and coverage settled in Data Preparation, the data is now genuinely comparable across markets and commodities. This section looks for the patterns the project's hypotheses actually depend on: seasonality, market-to-market differences, and any lagged relationship between weather and price.

Maize and beans anchor most of the analysis below, since they carry the deepest coverage and sit at the center of the business objectives from Section 1.

### 4.1 Univariate Analysis

In [ ]:
# distribution of price observations across commodities and markets
commodity_counts = prices["commodity"].value_counts().head(15)
market_counts = prices["market"].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

commodity_counts.plot(kind="barh", ax=axes[0], color="green")
axes[0].set_title("Top 15 Commodities by Number of Price Observations")
axes[0].invert_yaxis()

market_counts.plot(kind="barh", ax=axes[1], color="blue")
axes[1].set_title("Top 15 Markets by Number of Price Observations")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

Maize and beans dominate commodity coverage, and Nairobi sits alongside a handful of other major markets at the top of observation counts — confirming they carry enough depth for individual forecasting.

In [ ]:
# maize price distribution, IQR method
maize = prices[prices["commodity"] == "Maize"].copy()

q1 = maize["price"].quantile(0.25)
q3 = maize["price"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = maize[(maize["price"] < lower_bound) | (maize["price"] > upper_bound)]

print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}] KES")
print(f"Outlier observations detected: {len(outliers)} out of {len(maize)}")
outliers[["date", "market", "price"]].sort_values("price", ascending=False).head(10)

### 4.2 Bivariate Analysis

In [ ]:
# price spread: maize vs beans
priority_commodities = ["Maize", "Beans"]
subset = prices[prices["commodity"].isin(priority_commodities)]

plt.figure(figsize=(10, 5))
sns.boxplot(data=subset, x="commodity", y="price", hue="commodity", palette=["green", "brown"])
plt.title("Price Distribution: Maize vs Beans (KES)")
plt.ylabel("Price (KES)")
plt.xlabel("Commodity")
plt.show()

Beans trade at a consistently higher price point than maize, with a wider spread. Part of that gap reflects differing unit types across rows rather than a pure price difference, which is exactly what the unit standardization in Data Preparation was built to correct for before any modelling happens.

In [ ]:
# maize price coverage by market and year
pivot_check = maize.pivot_table(
    index="market",
    columns=maize["date"].dt.year,
    values="price",
    aggfunc="mean"
)

plt.figure(figsize=(16, 8))
sns.heatmap(pivot_check.isna(), cbar=False, cmap="Reds")
plt.title("Missing Maize Price Data by Market and Year (Red = Missing)")
plt.xlabel("Year")
plt.ylabel("Market")
plt.show()

Coverage is uneven across markets — some report almost every year, others show long red gaps. This is the same unevenness the historical-coverage classification in Data Preparation was built to handle, by grouping pairs into long-history and recent-only tracks rather than applying one blanket cutoff.

### 4.3 Multivariate / Relationship Analysis

The mechanic behind the early warning system depends on rainfall preceding price movement by a measurable lag — since harvests land months after planting rains, the relationship to test is never same-month, it's shifted forward. This section runs that test twice: an early check using the most readily available data, and a corrected version once weather and price are properly matched by market.

In [ ]:
# monthly maize price series, national average
maize_monthly = maize.groupby(pd.Grouper(key="date", freq="ME"))["price"].mean()

# sample daily weather for Nairobi, full year 2023
power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params_full_year = {
    "parameters": "T2M,PRECTOTCORR",
    "community": "ag",
    "longitude": nairobi_lon,
    "latitude": nairobi_lat,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}
response_year = requests.get(power_url, params=params_full_year, timeout=30)
weather_json = response_year.json()["properties"]["parameter"]

weather_df = pd.DataFrame({
    "date": pd.to_datetime(list(weather_json["T2M"].keys()), format="%Y%m%d"),
    "temperature": list(weather_json["T2M"].values()),
    "rainfall": list(weather_json["PRECTOTCORR"].values()),
})

In [ ]:
# rainfall-to-price correlation at 0 to 6 month lags
monthly_rainfall = weather_df.set_index("date")["rainfall"].resample("ME").sum()
combined = pd.DataFrame({"rainfall": monthly_rainfall, "price": maize_monthly}).dropna()

lag_results = {}
for lag in range(0, 7):
    shifted_rainfall = combined["rainfall"].shift(lag)
    lag_results[lag] = shifted_rainfall.corr(combined["price"])

lag_series = pd.Series(lag_results)

print("Correlation between rainfall (lagged) and maize price:")
print(lag_series.round(3))

lag_series.plot(kind="bar", figsize=(8, 4), color="#6A1B9A")
plt.title("Rainfall to Price Lag Correlation (Months)")
plt.xlabel("Lag (months)")
plt.ylabel("Correlation coefficient")
plt.show()

Lag 4 shows the strongest reading. But this uses one calendar year of Nairobi weather against a twenty-year national price series — a real mismatch in both geography and timeframe. The properly matched version of this test, using every shortlisted market and the full date range, happens in Feature Engineering, once the lagged features exist as master table columns.

### 4.4 Time-Series Analysis

In [ ]:
# national average maize price over time
plt.figure(figsize=(14, 5))
maize_monthly.plot(color="green", linewidth=1.8)
plt.title("National Average Maize Price Over Time (Monthly Mean)")
plt.ylabel("Price (KES)")
plt.xlabel("Date")
plt.show()

Maize prices show a clear long-term upward trend with repeated volatility spikes, not a stable plateau — consistent with the structural price instability described in Section 1.

In [ ]:
# average maize price by calendar month, all years combined
maize["month"] = maize["date"].dt.month
seasonal_avg = maize.groupby("month")["price"].mean()

plt.figure(figsize=(10, 5))
seasonal_avg.plot(kind="bar", color="orange")
plt.title("Average Maize Price by Calendar Month (All Years Combined)")
plt.xlabel("Month")
plt.ylabel("Average Price (KES)")
plt.xticks(rotation=0)
plt.show()

A mild, recurring seasonal shape is visible across the calendar year — early, if not yet conclusive, support for H1 on harvest-driven seasonality.

In [ ]:
# daily rainfall, Nairobi, full year 2023 -- checking for the known bimodal pattern
weather_df.set_index("date")[["rainfall"]].plot(figsize=(14, 4), color="blue")
plt.title("Daily Rainfall, Nairobi, 2023 (Checking for Bimodal Pattern)")
plt.ylabel("Rainfall (mm/day)")
plt.show()

Rainfall shows two distinct peak periods across the year, consistent with Kenya's known long-rains and short-rains pattern. That match against a well-documented real-world pattern is what makes the NASA POWER feed a credible source to build on, independent of how the price correlation test above turned out.

### 4.5 EDA Findings

- **H1 (seasonality):** mildly supported — a recurring calendar-month pattern is visible in national maize prices, though not sharp enough to call decisive on its own.
- **H2 (rainfall lag):** the preliminary single-market test suggests a possible relationship around a 4-month lag. Feature Engineering runs the properly matched, full-scale version of this test before the lag features are finalized.
- **Weather data quality:** independently credible. The bimodal rainfall pattern matches Kenya's known climate, so the NASA POWER feed is trustworthy as a source, even though rainfall alone doesn't explain price movement as hoped.
- **Coverage and scale:** commodity and market coverage is highly uneven, reinforcing why the tiered long-history/recent-only modelling approach from Data Preparation is necessary rather than optional.
- **Outliers:** a small number of extreme maize prices exist and are concentrated in specific markets rather than spread randomly — consistent with the refugee-camp cost dynamics already observed in Data Preparation's outlier review.

The weak weather correlation does not end the weather-as-feature question — it reframes it. Rainfall and temperature lags move into Feature Engineering as candidates the models get to actually evaluate, rather than as a relationship the notebook simply asserts.

## 5. Feature Engineering

### 5.1 Target Variable

In [ ]:
master = master.sort_values(["market", "commodity", "date"])
master["price_diff"] = master.groupby(["market", "commodity"])["price_per_kg"].diff()

print(f"Rows with a valid price_diff: {master['price_diff'].notna().sum()} out of {len(master)}")
print(f"Rows without one (first observation per pair): {master['price_diff'].isna().sum()}")

### 5.2 Lag Features

Weather is lagged on `weather_monthly`, not on `master` directly — `master` has one row per commodity per month, so a market can appear on several rows for the same date. Shifting there would pull values across commodities instead of across time. Shifting on `weather_monthly`, which has one row per market per month, then merging the result onto `master`, keeps each market's lag genuinely sequential.

In [ ]:
weather_monthly = weather_monthly.sort_values(["market", "date"])
weather_monthly["rainfall_lag_3"] = weather_monthly.groupby("market")["rainfall"].shift(3)
weather_monthly["rainfall_lag_4"] = weather_monthly.groupby("market")["rainfall"].shift(4)
weather_monthly["temperature_lag_3"] = weather_monthly.groupby("market")["temperature"].shift(3)
weather_monthly["temperature_lag_4"] = weather_monthly.groupby("market")["temperature"].shift(4)

lag_columns = ["market", "date", "rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]
weather_monthly["date_month"] = weather_monthly["date"].values.astype("datetime64[M]")

master = master.merge(
    weather_monthly[["market", "date_month", "rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]],
    on=["market", "date_month"],
    how="left"
)

print(f"Rows with matched lag features: {master['rainfall_lag_3'].notna().sum()} out of {len(master)}")

With the lag columns now genuinely present in `master`, matched by market and date rather than assumed, the relationship they were built to test can be checked properly for the first time — every shortlisted market, full history, correct match.

In [ ]:
rainfall_corr_3 = master["price_per_kg"].corr(master["rainfall_lag_3"])
rainfall_corr_4 = master["price_per_kg"].corr(master["rainfall_lag_4"])
temp_corr_3 = master["price_per_kg"].corr(master["temperature_lag_3"])
temp_corr_4 = master["price_per_kg"].corr(master["temperature_lag_4"])

print(f"Rainfall lag 3 months, correlation with price: {rainfall_corr_3:.3f}")
print(f"Rainfall lag 4 months, correlation with price: {rainfall_corr_4:.3f}")
print(f"Temperature lag 3 months, correlation with price: {temp_corr_3:.3f}")
print(f"Temperature lag 4 months, correlation with price: {temp_corr_4:.3f}")

The rainfall relationship the preliminary test suggested does not hold at full scale. Temperature shows a weak positive correlation at both lags, present but not strong on its own. Both are still carried forward as candidate features into the final feature set below — a weak linear correlation doesn't rule out a useful nonlinear contribution once a model sees price history alongside it, and that question belongs to Modelling, not to this test.

### 5.3 Weather Features

Rainfall (`PRECTOTCORR`) and temperature (`T2M`) are used only in their lagged form, never same-month. A harvest, and the price effect of that harvest, lands months after the rain that produced it — same-month weather would test the wrong mechanic entirely.

### 5.4 Feature Selection / Final Modelling Features

One feature list feeds every model in this project. Prophet and the LSTM tracks read the same four weather features from the same constant, so there is a single source of truth rather than two lists that could quietly drift apart.

In [ ]:
WEATHER_FEATURES = ["rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]

The LSTM tracks additionally need each market and commodity represented as a learnable entity, not just a text label. Encoders are fit on the full `shortlist`, not on `master`, since a handful of shortlisted pairs can still get filtered out of `master` entirely at the price-per-kg conversion step, and every pair the batch loop iterates over needs a valid entity id regardless of whether it's later skipped for having too few usable rows.

In [ ]:
market_encoder = LabelEncoder()
commodity_encoder = LabelEncoder()

market_encoder.fit(shortlist["market"])
commodity_encoder.fit(shortlist["commodity"])

master["market_id"] = master["market"].map(lambda m: market_encoder.transform([m])[0])
master["commodity_id"] = master["commodity"].map(lambda c: commodity_encoder.transform([c])[0])

n_markets = len(market_encoder.classes_)
n_commodities = len(commodity_encoder.classes_)

print(f"Unique markets encoded: {n_markets}")
print(f"Unique commodities encoded: {n_commodities}")

In [ ]:
FEATURES = ["price_diff"] + WEATHER_FEATURES
LOOKBACK = 6
MIN_ROWS = LOOKBACK + 4

print("Prophet regressors:", WEATHER_FEATURES, "| target: price_per_kg")
print("LSTM features:", FEATURES, "| target: price_diff")

### 5.5 Train/Validation/Test Preparation

Every split in this project is chronological, never random — the most recent months are always held out, so accuracy reflects real forward-looking performance rather than leakage from the future.

The version below splits the full master table at the 80th and 90th percentile of its date range as a reference. Modelling recomputes this same 80/10/10 chronological logic per series and per pair rather than applying it once globally, for the same reason completeness was measured per pair in Data Preparation — pairs don't share a start date, so one fixed cutoff would hand some pairs a generous training window and others almost none.

In [ ]:
master = master.sort_values("date")
cutoff_val = master["date"].quantile(0.8)
cutoff_test = master["date"].quantile(0.9)

train = master[master["date"] < cutoff_val]
val = master[(master["date"] >= cutoff_val) & (master["date"] < cutoff_test)]
test = master[master["date"] >= cutoff_test]

print(f"Train: {len(train)} rows, up to {train['date'].max()}")
print(f"Validation: {len(val)} rows, {val['date'].min()} to {val['date'].max()}")
print(f"Test: {len(test)} rows, from {test['date'].min()}")

## 6. Modelling

### 6.1 Modelling Objective

Three forecasting approaches are compared: a naive persistence baseline, Prophet, and LSTM (per-pair and pooled across entities). All three consume the same `WEATHER_FEATURES` from Feature Engineering. Every model is evaluated on the same chronologically held-out test period using MAE and MAPE, and judged against the naive baseline, not an arbitrary threshold, per the criterion set in Business Understanding.

Every stochastic step below — model initialization, training — is seeded, so the results in this section are the same on every rerun, not a fresh roll of the dice each time.

In [ ]:
# all modelling imports, in one place
import random
import numpy as np
import tensorflow as tf

from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Embedding, Flatten, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

FORECAST_HORIZON = 1

### 6.2 Naive Baseline Model

The baseline assumes next month's price equals the most recently observed price for that pair — operationalizing the one-month forecasting horizon this project targets, since the data's monthly resolution rules out a shorter one. A sophisticated model only earns its place if it beats this.

In [ ]:
baseline_data = master.sort_values(["market", "commodity", "date"]).copy()

baseline_data["naive_prediction"] = (
    baseline_data.groupby(["market", "commodity"])["price_per_kg"].shift(1)
)

baseline_test = baseline_data.dropna(subset=["price_per_kg", "naive_prediction"]).copy()

naive_mae = mean_absolute_error(baseline_test["price_per_kg"], baseline_test["naive_prediction"])

non_zero = baseline_test["price_per_kg"] != 0
naive_mape = np.mean(
    np.abs(
        (baseline_test.loc[non_zero, "price_per_kg"] - baseline_test.loc[non_zero, "naive_prediction"])
        / baseline_test.loc[non_zero, "price_per_kg"]
    )
) * 100

print(f"Naive Baseline MAE: {naive_mae:.2f}")
print(f"Naive Baseline MAPE: {naive_mape:.2f}%")

### 6.3 Prophet

The pipeline is built and validated on one well-populated series before scaling to the full shortlist. Kitui Maize (white) has 180 monthly observations — deep enough history to develop against, and a fair proving ground before Section 6.5 runs the same logic across all 133 pairs.

In [ ]:
market = "Kitui"
commodity = "Maize (white)"

series = master[(master["market"] == market) & (master["commodity"] == commodity)].copy()
series = series.sort_values("date")

print(f"Observations for {commodity} in {market}: {len(series)}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(series["date"], series["price_per_kg"], label="Actual Price")
plt.title(f"{commodity} Price in {market}")
plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

Prophet needs a date column named `ds` and a target named `y`. It reads the same `WEATHER_FEATURES` defined in Feature Engineering as external regressors — the identical four columns the LSTM tracks use, just serving a different role for a different algorithm.

In [ ]:
prophet_data = series[["date", "price_per_kg"] + WEATHER_FEATURES].copy()
prophet_data = prophet_data.rename(columns={"date": "ds", "price_per_kg": "y"})
prophet_data = prophet_data.dropna()

series_train_end = series["date"].quantile(0.8)
series_val_end = series["date"].quantile(0.9)

prophet_train = prophet_data[prophet_data["ds"] <= series_train_end].copy()
prophet_val = prophet_data[(prophet_data["ds"] > series_train_end) & (prophet_data["ds"] <= series_val_end)].copy()
prophet_test = prophet_data[prophet_data["ds"] > series_val_end].copy()

print(f"Train: {len(prophet_train)}, Validation: {len(prophet_val)}, Test: {len(prophet_test)}")

In [ ]:
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)

for feature in WEATHER_FEATURES:
    prophet_model.add_regressor(feature)

prophet_model.fit(prophet_train)

In [ ]:
prophet_val_forecast = prophet_model.predict(prophet_val[["ds"] + WEATHER_FEATURES])
prophet_val_results = prophet_val[["ds", "y"]].copy()
prophet_val_results["prediction"] = prophet_val_forecast["yhat"].values

val_mae = mean_absolute_error(prophet_val_results["y"], prophet_val_results["prediction"])
non_zero = prophet_val_results["y"] != 0
val_mape = np.mean(
    np.abs((prophet_val_results.loc[non_zero, "y"] - prophet_val_results.loc[non_zero, "prediction"]) / prophet_val_results.loc[non_zero, "y"])
) * 100

print(f"Prophet Validation MAE: {val_mae:.2f}")
print(f"Prophet Validation MAPE: {val_mape:.2f}%")

In [ ]:
prophet_test_forecast = prophet_model.predict(prophet_test[["ds"] + WEATHER_FEATURES])
prophet_test_results = prophet_test[["ds", "y"]].copy()
prophet_test_results["prediction"] = prophet_test_forecast["yhat"].values

prophet_test_mae = mean_absolute_error(prophet_test_results["y"], prophet_test_results["prediction"])
non_zero = prophet_test_results["y"] != 0
prophet_test_mape = np.mean(
    np.abs((prophet_test_results.loc[non_zero, "y"] - prophet_test_results.loc[non_zero, "prediction"]) / prophet_test_results.loc[non_zero, "y"])
) * 100

print(f"Prophet Test MAE: {prophet_test_mae:.2f}")
print(f"Prophet Test MAPE: {prophet_test_mape:.2f}%")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(prophet_test_results["ds"], prophet_test_results["y"], label="Actual")
plt.plot(prophet_test_results["ds"], prophet_test_results["prediction"], label="Prophet")
plt.title(f"Prophet Test Forecast: {commodity} - {market}")
plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

### 6.4 LSTM

The LSTM reads exactly the `FEATURES` list defined in Feature Engineering — `price_diff` plus the same four weather features Prophet just used. Predicting the differenced price rather than the level keeps the series stable enough for a single scaler to handle, which matters here for the same reason it mattered in Feature Engineering: this pipeline needs to generalize to the pooled model in Section 6.5, where many pairs share one scaler.

Because the target is now a difference, evaluating it in comparable price terms takes one extra step: each predicted difference is added back onto the last known actual price before MAE and MAPE are computed, so this stays comparable to the naive baseline and Prophet above.

In [ ]:
lstm_data = series[["date", "price_per_kg"] + FEATURES].dropna().sort_values("date")

train_end = lstm_data["date"].quantile(0.8)
val_end = lstm_data["date"].quantile(0.9)

lstm_train = lstm_data[lstm_data["date"] <= train_end].copy()
lstm_val = lstm_data[(lstm_data["date"] > train_end) & (lstm_data["date"] <= val_end)].copy()
lstm_test = lstm_data[lstm_data["date"] > val_end].copy()

scaler = MinMaxScaler()
scaler.fit(lstm_train[FEATURES])

train_scaled = scaler.transform(lstm_train[FEATURES])
val_scaled = scaler.transform(lstm_val[FEATURES])
test_scaled = scaler.transform(lstm_test[FEATURES])

print(f"Train: {len(lstm_train)}, Validation: {len(lstm_val)}, Test: {len(lstm_test)}")

In [ ]:
def create_sequences(data, lookback):
    """Convert a scaled series into rolling input sequences and next-step targets."""
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        y.append(data[i, 0])
    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, LOOKBACK)
X_val, y_val = create_sequences(val_scaled, LOOKBACK)
X_test, y_test = create_sequences(test_scaled, LOOKBACK)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
lstm_model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
lstm_predictions_scaled = lstm_model.predict(X_test).flatten()

prediction_matrix = np.zeros((len(lstm_predictions_scaled), len(FEATURES)))
prediction_matrix[:, 0] = lstm_predictions_scaled
predicted_diff = scaler.inverse_transform(prediction_matrix)[:, 0]

# reconstruct price level: last actual price + predicted change
previous_prices = lstm_test["price_per_kg"].values[LOOKBACK - 1:-1]
actual_prices = lstm_test["price_per_kg"].values[LOOKBACK:]
predicted_prices = previous_prices + predicted_diff

lstm_mae = mean_absolute_error(actual_prices, predicted_prices)
non_zero = actual_prices != 0
lstm_mape = np.mean(np.abs((actual_prices[non_zero] - predicted_prices[non_zero]) / actual_prices[non_zero])) * 100

print(f"LSTM Test MAE: {lstm_mae:.2f}")
print(f"LSTM Test MAPE: {lstm_mape:.2f}%")

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Naive Baseline", "Prophet", "LSTM"],
    "MAE": [naive_mae, prophet_test_mae, lstm_mae],
    "MAPE": [naive_mape, prophet_test_mape, lstm_mape]
})

model_comparison = model_comparison.sort_values("MAE").reset_index(drop=True)
model_comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(model_comparison["Model"], model_comparison["MAE"])
plt.title("Model Comparison — MAE (Kitui Maize, white)")
plt.xlabel("Model")
plt.ylabel("Mean Absolute Error")
plt.show()

For this single series, LSTM is the clear winner on both metrics, and both Prophet and LSTM beat the naive baseline on MAE. Whether that holds up is exactly what Section 6.5 tests — one series doing well is a promising sign, not proof, and 133 pairs is where that claim actually gets tested.

### 6.5 Batch / Global Modelling

The single-series pipeline validated above is now applied across the full shortlist. Every pair reuses the `FEATURES`, `WEATHER_FEATURES`, `market_id`, and `commodity_id` already established in Feature Engineering — nothing gets re-encoded or redefined here.

The model track assigned in Data Preparation determines the comparison structure: long-history pairs went to Prophet, but every LSTM-track pair, and in fact every pair regardless of track, gets evaluated through the pooled architecture built here, so a pair's `price_diff` behaves consistently whichever track it was assigned to for its stand-alone model.

#### 6.5.1 Build Per-Pair Sequences

Each pair gets its own chronological train/validation/test split and its own `MinMaxScaler`, fit only on that pair's training rows — pooling many series into one model doesn't mean pooling their scales. `price_diff` is read directly from `master`, where Feature Engineering already computed it; it is not recomputed here.

Raw, unscaled last-known-price and actual-price arrays are kept for both validation and test, so a per-pair validation score can be computed later without the test set ever entering that decision.

In [ ]:
def build_pair_sequences(pair_df, market_id, commodity_id, model_track, lookback=LOOKBACK):
    pair_df = pair_df.sort_values("date").copy()
    pair_df = pair_df.dropna(subset=FEATURES)

    if len(pair_df) < MIN_ROWS:
        return None

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)

    train_mask = pair_df["date"] <= train_end
    val_mask = (pair_df["date"] > train_end) & (pair_df["date"] <= val_end)
    test_mask = pair_df["date"] > val_end

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return None

    naive_actual = pair_df.loc[naive_ready, "price_per_kg"]
    naive_pred_vals = naive_pred.loc[naive_ready]
    naive_mae = mean_absolute_error(naive_actual, naive_pred_vals)
    naive_mape = np.mean(np.abs((naive_actual - naive_pred_vals) / naive_actual)) * 100

    val_naive_ready = val_mask & naive_pred.notna()
    if val_naive_ready.any():
        val_naive_actual = pair_df.loc[val_naive_ready, "price_per_kg"]
        val_naive_pred_vals = naive_pred.loc[val_naive_ready]
        val_naive_mae = mean_absolute_error(val_naive_actual, val_naive_pred_vals)
    else:
        val_naive_mae = np.nan

    scaler = MinMaxScaler()
    scaler.fit(pair_df.loc[train_mask, FEATURES])

    train_scaled = scaler.transform(pair_df.loc[train_mask, FEATURES])
    val_scaled = scaler.transform(pair_df.loc[val_mask, FEATURES])
    test_scaled = scaler.transform(pair_df.loc[test_mask, FEATURES])

    if len(train_scaled) <= lookback:
        return None

    X_train, y_train = create_sequences(train_scaled, lookback)

    val_source = np.vstack([train_scaled[-lookback:], val_scaled]) if len(val_scaled) > 0 else train_scaled[-lookback:]
    X_val, y_val = create_sequences(val_source, lookback)

    test_source = np.vstack([val_scaled[-lookback:], test_scaled]) if len(val_scaled) >= lookback else np.vstack([train_scaled[-lookback:], test_scaled])
    X_test, y_test = create_sequences(test_source, lookback)

    if len(X_train) < 1 or len(X_test) == 0:
        return None

    last_price_val = pair_df["price_per_kg"].shift(1).loc[val_mask].values
    actual_price_val = pair_df.loc[val_mask, "price_per_kg"].values
    last_price_test = pair_df["price_per_kg"].shift(1).loc[test_mask].values
    actual_price_test = pair_df.loc[test_mask, "price_per_kg"].values

    return {
        "market": pair_df["market"].iloc[0], "commodity": pair_df["commodity"].iloc[0],
        "market_id": market_id, "commodity_id": commodity_id, "model_track": model_track,
        "scaler": scaler, "naive_mae": naive_mae, "naive_mape": naive_mape, "val_naive_mae": val_naive_mae,
        "X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val, "X_test": X_test, "y_test": y_test,
        "last_price_val": last_price_val, "actual_price_val": actual_price_val,
        "last_price_test": last_price_test, "actual_price_test": actual_price_test,
    }

Pairs that fail the minimum row requirement, end up with no usable test sequences, or have no naive-ready test rows return `None` and are skipped when the batch is assembled below.

In [ ]:
pair_bundles = []
X_train_list, y_train_list, mkt_train_list, com_train_list = [], [], [], []
X_val_list, y_val_list, mkt_val_list, com_val_list = [], [], [], []
X_test_list, y_test_list, mkt_test_list, com_test_list = [], [], [], []
pair_idx_val_list, pair_idx_test_list = [], []
last_price_val_list, actual_price_val_list = [], []
last_price_test_list, actual_price_test_list = [], []

for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]
    market_id = market_encoder.transform([row["market"]])[0]
    commodity_id = commodity_encoder.transform([row["commodity"]])[0]

    bundle = build_pair_sequences(pair_df, market_id, commodity_id, row["model_track"])
    if bundle is None:
        continue

    pair_idx = len(pair_bundles)
    pair_bundles.append(bundle)

    n_tr, n_va, n_te = len(bundle["X_train"]), len(bundle["X_val"]), len(bundle["X_test"])

    X_train_list.append(bundle["X_train"]); y_train_list.append(bundle["y_train"])
    mkt_train_list.append(np.full(n_tr, market_id)); com_train_list.append(np.full(n_tr, commodity_id))

    X_val_list.append(bundle["X_val"]); y_val_list.append(bundle["y_val"])
    mkt_val_list.append(np.full(n_va, market_id)); com_val_list.append(np.full(n_va, commodity_id))
    pair_idx_val_list.append(np.full(n_va, pair_idx))
    last_price_val_list.append(bundle["last_price_val"]); actual_price_val_list.append(bundle["actual_price_val"])

    X_test_list.append(bundle["X_test"]); y_test_list.append(bundle["y_test"])
    mkt_test_list.append(np.full(n_te, market_id)); com_test_list.append(np.full(n_te, commodity_id))
    pair_idx_test_list.append(np.full(n_te, pair_idx))
    last_price_test_list.append(bundle["last_price_test"]); actual_price_test_list.append(bundle["actual_price_test"])

X_train_all = np.concatenate(X_train_list); y_train_all = np.concatenate(y_train_list)
market_train_all = np.concatenate(mkt_train_list); commodity_train_all = np.concatenate(com_train_list)

X_val_all = np.concatenate(X_val_list); y_val_all = np.concatenate(y_val_list)
market_val_all = np.concatenate(mkt_val_list); commodity_val_all = np.concatenate(com_val_list)
pair_idx_val_all = np.concatenate(pair_idx_val_list)
last_price_val_all = np.concatenate(last_price_val_list); actual_price_val_all = np.concatenate(actual_price_val_list)

X_test_all = np.concatenate(X_test_list); y_test_all = np.concatenate(y_test_list)
market_test_all = np.concatenate(mkt_test_list); commodity_test_all = np.concatenate(com_test_list)
pair_idx_test_all = np.concatenate(pair_idx_test_list)
last_price_test_all = np.concatenate(last_price_test_list); actual_price_test_all = np.concatenate(actual_price_test_list)

print(f"Pairs included: {len(pair_bundles)} out of {len(shortlist)}")

#### 6.5.2 Entity Embedding LSTM

One shared model trains across every pair at once. It takes three inputs: the numeric sequence window, the market id, and the commodity id. Market and commodity each get an `Embedding` layer mapping their integer id to a learned dense vector, concatenated with the LSTM's output before the final dense layers.

Every learnable block carries its own regularization — recurrent dropout and an L2 penalty on the LSTM, L2 on both embedding tables, dropout before and after the merge. An unconstrained LSTM and embedding table have more than enough capacity to memorize which specific market or commodity a sequence belongs to rather than learning something that transfers across series, which is the entire point of pooling in the first place.

In [ ]:
n_features = len(FEATURES)
market_embed_dim = min(8, (n_markets + 1) // 2)
commodity_embed_dim = min(8, (n_commodities + 1) // 2)

sequence_input = Input(shape=(LOOKBACK, n_features), name="sequence_input")
market_input = Input(shape=(1,), name="market_input")
commodity_input = Input(shape=(1,), name="commodity_input")

market_embed = Embedding(n_markets, market_embed_dim, embeddings_regularizer=l2(0.01), name="market_embedding")(market_input)
market_embed = Flatten()(market_embed)

commodity_embed = Embedding(n_commodities, commodity_embed_dim, embeddings_regularizer=l2(0.01), name="commodity_embedding")(commodity_input)
commodity_embed = Flatten()(commodity_embed)

lstm_out = LSTM(16, return_sequences=False, recurrent_dropout=0.2, kernel_regularizer=l2(0.01))(sequence_input)
lstm_out = Dropout(0.3)(lstm_out)

merged = Concatenate()([lstm_out, market_embed, commodity_embed])
dense_out = Dense(16, activation="relu", kernel_regularizer=l2(0.01))(merged)
dense_out = Dropout(0.3)(dense_out)
output = Dense(1)(dense_out)

embedding_model = Model(inputs=[sequence_input, market_input, commodity_input], outputs=output)
embedding_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
embedding_model.summary()

Embedding dimensions are capped at 8, roughly half the entity count. LSTM units are cut to 16, far below the 64 used for the single-series demo — a smaller model that generalizes across many short, noisy series is worth more here than a larger one that memorizes them individually. The L2 values (0.01) and dropout rates (0.2–0.3) are reasonable starting points, not tuned optima.

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

embedding_history = embedding_model.fit(
    [X_train_all, market_train_all, commodity_train_all], y_train_all,
    validation_data=([X_val_all, market_val_all, commodity_val_all], y_val_all),
    epochs=100, batch_size=32, callbacks=[early_stopping], verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(embedding_history.history["loss"], label="Training Loss")
plt.plot(embedding_history.history["val_loss"], label="Validation Loss")
plt.title("Entity Embedding LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

This is one training curve for every pair combined, not a separate curve per pair. A converging validation loss here would indicate the shared representation is generalizing across markets and commodities, not memorizing one series.

#### 6.5.3 Global Model Without Entity Identity

The entity-embedding model above tells the network which market and commodity each row belongs to. This isolates a separate question: is the improvement over independent per-pair models coming from pooling the data itself, or from the embeddings letting the model condition on identity?

This version uses the identical pooled training batch, lookback window, and LSTM capacity and regularization as the entity-embedding model, but receives only the numeric sequence — every row across every pair treated as one generic series, no market or commodity identifier at all. If it performs close to the entity-embedding model, pooling more data is what matters and the embeddings add little. If it performs closer to the independent per-pair models in 6.5.5, entity identity is doing real work.

In [ ]:
global_sequence_input = Input(shape=(LOOKBACK, n_features), name="global_sequence_input")

global_lstm_out = LSTM(16, return_sequences=False, recurrent_dropout=0.2, kernel_regularizer=l2(0.01))(global_sequence_input)
global_lstm_out = Dropout(0.3)(global_lstm_out)

global_dense_out = Dense(16, activation="relu", kernel_regularizer=l2(0.01))(global_lstm_out)
global_dense_out = Dropout(0.3)(global_dense_out)

global_output = Dense(1)(global_dense_out)

global_model = Model(inputs=global_sequence_input, outputs=global_output, name="global_model")
global_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
global_model.summary()

Embedding dimensions are capped at 8, roughly half the entity count. LSTM units are cut to 16, far below the 64 used for the single-series demo — a smaller model that generalizes across many short, noisy series is worth more here than a larger one that memorizes them individually. The L2 values (0.01) and dropout rates (0.2–0.3) are reasonable starting points, not tuned optima.

In [ ]:
global_early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

global_history = global_model.fit(
    X_train_all, y_train_all,
    validation_data=(X_val_all, y_val_all),
    epochs=100, batch_size=32, callbacks=[global_early_stopping], verbose=1
)

In [ ]:
predicted_val_delta_global = global_model.predict(X_val_all, verbose=0).flatten()
predicted_test_delta_global = global_model.predict(X_test_all, verbose=0).flatten()

global_results = []

for pair_idx, bundle in enumerate(pair_bundles):
    test_mask_rows = pair_idx_test_all == pair_idx
    if not test_mask_rows.any():
        continue

    scaler = bundle["scaler"]
    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta_global[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]

    pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    actual = actual_price_test_all[test_mask_rows]

    mae = mean_absolute_error(actual, pred)
    nz = actual != 0
    mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100

    global_results.append({
        "market": bundle["market"], "commodity": bundle["commodity"], "model_track": bundle["model_track"],
        "model_mae": mae, "model_mape": mape,
        "naive_mae": bundle["naive_mae"], "naive_mape": bundle["naive_mape"],
    })

global_results = pd.DataFrame(global_results)
global_results["beats_naive"] = global_results["model_mae"] < global_results["naive_mae"]

print(f"Global model, all pairs: MAE {global_results['model_mae'].mean():.2f}, MAPE {global_results['model_mape'].mean():.2f}%")
print(f"Naive baseline, same pairs: MAE {global_results['naive_mae'].mean():.2f}, MAPE {global_results['naive_mape'].mean():.2f}%")
print(f"Pairs where global model beats naive: {global_results['beats_naive'].mean() * 100:.1f}%")
print()
print("Global model by track:")
print(global_results.groupby("model_track")[["model_mae", "model_mape"]].mean())

The identity-free global model beats naive on only 13.6% of pairs, well below what the entity-embedding version achieves once routed through validation in 6.5.4. Entity identity is doing real work here, not just riding on pooled data volume.

#### 6.5.4 Per-Pair Model Selection

Rather than a hard switch between the model's forecast and naive, each pair's final forecast is a weighted blend of the two, with the weight set by how much validation evidence supports the model and how strong that evidence was. This follows a well-established result in forecasting research, going back to Bates and Granger (1969) and repeatedly confirmed in the M-competitions, that combining forecasts tends to outperform confidently picking one, especially when the evidence for picking is thin.

A pair with only 4 validation rows and an apparent win gets nudged only slightly away from naive. A pair with a longer, more consistent validation record gets weighted more heavily toward the model. A pair with no real evidence for the model gets a weight of exactly 0 — mathematically identical to pure naive — so this design can never do worse than a hard-switch router by construction, only more cautious.

In [ ]:
MIN_VAL_ROWS_FOR_ANY_WEIGHT = 4
FULL_CONFIDENCE_VAL_ROWS = 8

def compute_model_weight(val_rows, val_naive_mae, val_model_mae):
    if val_rows < MIN_VAL_ROWS_FOR_ANY_WEIGHT or np.isnan(val_naive_mae) or val_naive_mae == 0:
        return 0.0
    improvement = (val_naive_mae - val_model_mae) / val_naive_mae
    confidence = min(val_rows / FULL_CONFIDENCE_VAL_ROWS, 1.0)
    return max(0.0, min(improvement, 1.0)) * confidence

predicted_val_delta = embedding_model.predict([X_val_all, market_val_all, commodity_val_all], verbose=0).flatten()
predicted_test_delta = embedding_model.predict([X_test_all, market_test_all, commodity_test_all], verbose=0).flatten()

selection_results = []

for pair_idx, bundle in enumerate(pair_bundles):
    val_mask_rows = pair_idx_val_all == pair_idx
    test_mask_rows = pair_idx_test_all == pair_idx
    if not test_mask_rows.any():
        continue

    scaler = bundle["scaler"]
    weight = 0.0

    if val_mask_rows.sum() >= MIN_VAL_ROWS_FOR_ANY_WEIGHT and not np.isnan(bundle["val_naive_mae"]):
        val_diff_matrix = np.zeros((val_mask_rows.sum(), n_features))
        val_diff_matrix[:, 0] = predicted_val_delta[val_mask_rows]
        val_predicted_diff = scaler.inverse_transform(val_diff_matrix)[:, 0]
        val_pred = last_price_val_all[val_mask_rows] + val_predicted_diff
        val_actual = actual_price_val_all[val_mask_rows]
        val_model_mae = mean_absolute_error(val_actual, val_pred)
        weight = compute_model_weight(val_mask_rows.sum(), bundle["val_naive_mae"], val_model_mae)

    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    model_pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    naive_pred_test = last_price_test_all[test_mask_rows]
    actual = actual_price_test_all[test_mask_rows]

    blended_pred = weight * model_pred + (1 - weight) * naive_pred_test

    final_mae = mean_absolute_error(actual, blended_pred)
    nz = actual != 0
    final_mape = np.mean(np.abs((actual[nz] - blended_pred[nz]) / actual[nz])) * 100

    label = "model" if weight >= 0.5 else ("blend" if weight > 0 else "naive")

    selection_results.append({
        "market": bundle["market"], "commodity": bundle["commodity"], "model_track": bundle["model_track"],
        "weight": weight, "chosen_forecast": label,
        "final_mae": final_mae, "final_mape": final_mape, "naive_mae": bundle["naive_mae"],
        "naive_forecast": naive_pred_test[-1], "model_forecast": model_pred[-1], "blended_forecast": blended_pred[-1],
    })

selection_results = pd.DataFrame(selection_results)
selection_results["beats_naive"] = selection_results["final_mae"] <= selection_results["naive_mae"]

In [ ]:
print(f"Pairs with zero model weight (pure naive): {(selection_results['weight'] == 0).sum()}")
print(f"Pairs with partial weight (blend): {((selection_results['weight'] > 0) & (selection_results['weight'] < 0.5)).sum()}")
print(f"Pairs with weight >= 0.5 (model-leaning): {(selection_results['weight'] >= 0.5).sum()}")
print(f"Overall mean MAE with blending: {selection_results['final_mae'].mean():.2f}")
print(f"Overall mean MAPE with blending: {selection_results['final_mape'].mean():.2f}%")
print(f"Pairs at or better than naive: {selection_results['beats_naive'].mean() * 100:.1f}%")

In [ ]:
print("Distribution of model weight across all pairs:")
print(selection_results["weight"].describe())

No pair ever crosses a 0.5 weight — the model never fully overrides naive anywhere in the shortlist. By construction, every pair routed to naive ties naive exactly, so the 82.6% beat-rate above is only meaningful once checked against how the 25 blended pairs actually performed on the untouched test set, which is exactly what Evaluation revisits.

#### 6.5.5 Pooling Versus Per-Pair Models on the Short-History Track

The pooled entity-embedding model is only worth its added complexity if it outperforms fitting an independent model per pair — specifically on the `lstm` track, the short-history series pooling was designed to help. This rebuilds the original per-pair approach, deliberately shrunk to a comparable capacity, and compares it against the pooled model's own performance on the same pairs, before the 6.5.4 routing layer is applied, so the two improvements aren't blurred together.

In [ ]:
def evaluate_pair(market, commodity, model_track, min_rows=10, lookback=6):
    pair = master[(master["market"] == market) & (master["commodity"] == commodity)].copy()
    pair = pair.sort_values("date").dropna(subset=["price_per_kg"] + WEATHER_FEATURES)

    if len(pair) < min_rows:
        return None

    train_end = pair["date"].quantile(0.8)
    val_end = pair["date"].quantile(0.9)

    pair["naive_pred"] = pair["price_per_kg"].shift(1)
    test_naive = pair[pair["date"] > val_end].dropna(subset=["naive_pred"])
    if test_naive.empty:
        return None
    naive_mae = mean_absolute_error(test_naive["price_per_kg"], test_naive["naive_pred"])
    naive_mape = np.mean(np.abs((test_naive["price_per_kg"] - test_naive["naive_pred"]) / test_naive["price_per_kg"])) * 100

    feats = ["price_per_kg"] + WEATHER_FEATURES
    train_m = pair["date"] <= train_end
    val_m = (pair["date"] > train_end) & (pair["date"] <= val_end)
    test_m = pair["date"] > val_end

    scaler_p = MinMaxScaler()
    scaler_p.fit(pair.loc[train_m, feats])
    train_s = scaler_p.transform(pair.loc[train_m, feats])
    val_s = scaler_p.transform(pair.loc[val_m, feats])
    test_s = scaler_p.transform(pair.loc[test_m, feats])

    if len(train_s) <= lookback:
        return None

    X_tr, y_tr = create_sequences(train_s, lookback)
    val_source = np.vstack([train_s[-lookback:], val_s]) if len(val_s) > 0 else train_s[-lookback:]
    X_va, y_va = create_sequences(val_source, lookback)
    test_source = np.vstack([val_s[-lookback:], test_s]) if len(val_s) >= lookback else np.vstack([train_s[-lookback:], test_s])
    X_te, y_te = create_sequences(test_source, lookback)

    if len(X_tr) < 1 or len(X_te) == 0:
        return None

    try:
        model = Sequential([
            LSTM(8, input_shape=(X_tr.shape[1], X_tr.shape[2]), return_sequences=False, recurrent_dropout=0.2),
            Dropout(0.3),
            Dense(1)
        ])
        model.compile(optimizer="adam", loss="mse")
        es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=40, batch_size=4, callbacks=[es], verbose=0)

        pred_scaled = model.predict(X_te, verbose=0).flatten()
        pm = np.zeros((len(pred_scaled), len(feats))); pm[:, 0] = pred_scaled
        pred = scaler_p.inverse_transform(pm)[:, 0]

        am = np.zeros((len(y_te), len(feats))); am[:, 0] = y_te
        actual = scaler_p.inverse_transform(am)[:, 0]

        mae = mean_absolute_error(actual, pred)
        nz = actual != 0
        mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    except Exception as e:
        return {"market": market, "commodity": commodity, "n_test": len(test_naive),
                "naive_mae": naive_mae, "naive_mape": naive_mape, "model_mae": np.nan, "model_mape": np.nan, "error": str(e)}

    return {"market": market, "commodity": commodity, "n_test": len(test_naive),
            "naive_mae": naive_mae, "naive_mape": naive_mape, "model_mae": mae, "model_mape": mape}

In [ ]:
lstm_shortlist = shortlist[shortlist["model_track"] == "lstm"]

per_pair_lstm_results = []
for _, row in lstm_shortlist.iterrows():
    res = evaluate_pair(row["market"], row["commodity"], row["model_track"])
    if res is not None:
        per_pair_lstm_results.append(res)

per_pair_lstm_results = pd.DataFrame(per_pair_lstm_results)
per_pair_lstm_results["beats_naive"] = per_pair_lstm_results["model_mae"] < per_pair_lstm_results["naive_mae"]

In [ ]:
pooled_only_results = []
for pair_idx, bundle in enumerate(pair_bundles):
    test_mask_rows = pair_idx_test_all == pair_idx
    if not test_mask_rows.any():
        continue
    scaler = bundle["scaler"]
    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    actual = actual_price_test_all[test_mask_rows]
    mae = mean_absolute_error(actual, pred)
    nz = actual != 0
    mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    pooled_only_results.append({"market": bundle["market"], "commodity": bundle["commodity"],
                                 "model_track": bundle["model_track"], "pooled_mae": mae, "pooled_mape": mape})

pooled_only_results = pd.DataFrame(pooled_only_results)
by_track = pooled_only_results.groupby("model_track")[["pooled_mae", "pooled_mape"]].mean()

per_pair_mae = per_pair_lstm_results["model_mae"].mean()
per_pair_mape = per_pair_lstm_results["model_mape"].mean()
per_pair_beat_rate = per_pair_lstm_results["beats_naive"].mean() * 100
pooled_lstm = by_track.loc["lstm"]

print(f"Per-pair LSTM, lstm-track only: MAE {per_pair_mae:.2f}, MAPE {per_pair_mape:.2f}%, beat-naive {per_pair_beat_rate:.1f}%")
print(f"Pooled embedding, lstm-track only (before routing): MAE {pooled_lstm['pooled_mae']:.2f}, MAPE {pooled_lstm['pooled_mape']:.2f}%")

Pooling roughly halves the error on the short-history track — 15.89 down to 6.25 MAE. This is the strongest single result in the modelling section: entity identity plus shared training genuinely helps where a pair's own history is too short to train a standalone model well.

#### 6.5.6 Diagnosing Excluded Pairs

Only 1 of the 133 shortlisted pairs never made it into `pair_bundles`. This checks that pair against the same guard clauses in `build_pair_sequences` directly, rather than assuming a cause.

In [ ]:
def diagnose_pair(pair_df, lookback=LOOKBACK, min_rows=MIN_ROWS):
    pair_df = pair_df.sort_values("date").copy()
    before = len(pair_df)
    pair_df = pair_df.dropna(subset=FEATURES)
    after = len(pair_df)

    if after < min_rows:
        return f"too few rows after dropna ({after} rows, started at {before})"

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)
    train_mask = pair_df["date"] <= train_end
    test_mask = pair_df["date"] > val_end

    if train_mask.sum() <= lookback:
        return f"train rows ({train_mask.sum()}) too few for lookback ({lookback})"

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return "no usable test rows for naive baseline"

    return "ok"

diagnosis = []
for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]
    reason = diagnose_pair(pair_df)
    diagnosis.append({"market": row["market"], "commodity": row["commodity"], "model_track": row["model_track"], "reason": reason})

diagnosis = pd.DataFrame(diagnosis)
excluded = diagnosis[diagnosis["reason"] != "ok"]

print(excluded["reason"].value_counts())
print()
print(excluded.groupby("model_track")["reason"].value_counts())

One long-history pair loses every row once the weather features are required, which points to a weather retrieval gap for that specific market rather than a price data problem — the same single-market weather gap already surfaced in Data Preparation's 98.98% match rate.